In [0]:
#SETTING AND CONFIG CELL
import math
import csv
import sys
import re
import sqlite3 #reads nsys profiles exported to sqlite
import statistics
import subprocess #runs the harness / nsys from the MAKE DATA cells
from pathlib import Path
import matplotlib.pyplot as plot

#NOTE: the notebook runs from its own folder (YOLO/src/analysis/graphs), so the logs are 3 folders up in YOLO/benchmarking

BENCH_DIR = Path.cwd().parent.parent.parent / "benchmarking"
GRAPH_DIR = BENCH_DIR / "graphs" #location to save graphs to
GRAPH_DIR.mkdir(parents = True, exist_ok = True) #make the graphs folder if it doesn't exist yet
HARNESS_DIR = Path.cwd().parent.parent / "substage_compiled" #harness main.py, used by the MAKE DATA cells
PLATFORM = "Jetson AGX Orin (CUDA)" #only used in graph titles

#adjust configuration of desired run
BATCH_SIZE = 1
STEP = None
OPTIMIZATION = None #e.g. "inductor"
PRECISION = None    #e.g. "bf16" or "fp32"

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#LATENCY HELPER FUNCS

#read stage_timings.csv, one dict per row
#NOTE: rows are per image times in ms, columns:
#      timestamp,script,pass,device,batch_size,num_images,stage,substep,ms,step,optimization,precision

rows = list(csv.DictReader(open(BENCH_DIR / "stage_timings.csv")))
print(f"{len(rows)} rows in {BENCH_DIR / 'stage_timings.csv'}")
stages = {"pre_processing" : "#2a78d6", "inference" : "#eb6834", "post_processing" : "#1baf7a"} #dict to store each stage and their 
                                                                                                #associated color scheme

#helper func to find a run with desired configuration
def wanted_row(row):
    return ((BATCH_SIZE is None or row.get("batch_size") == str(BATCH_SIZE)) and (STEP is None or row.get("step") == str(STEP))
            and (OPTIMIZATION is None or row.get("optimization") == str(OPTIMIZATION)) and (PRECISION is None or row.get("precision") == PRECISION))

#helper func to find latest run
#NOTE: accepts pass_name as the type of harness run/pass associated with stored data point (e.g. substep_timing for substages,
#                                                                                           or wall_timing for stage wall times,
#                                                                                           or e2e for end to end batch times)
def latest_run(pass_name):
    matches = [r for r in rows if r["pass"] == pass_name and wanted_row(r)]
    if not matches:
        raise ValueError(f"NO {pass_name} ROWS MATCH THE SETTINGS (batch {BATCH_SIZE}, step {STEP}, opt {OPTIMIZATION}, precision {PRECISION})")
    latest = max(r["timestamp"] for r in matches) #finx most recent timestamp
    return [r for r in matches if r["timestamp"]==latest] #only grabs rows from most recent run

In [ ]:
#NOTE: CREATES A GRAPH THAT SHOWS ASYNC LATENCY BY SUBSTAGE (PER IMAGE) VIA CUDA EVENTS

#rows of most recent cuda events recorded in stage_timing.csv
run = [r for r in latest_run("substep_timing") if r["substep"] != "SUM_OF_STEPS"] #substeps only, in pipeline order
first = run[0] #grabs the first row taht matches cuda events
total = sum(float(r["ms"]) for r in run) #total time to find stage share percentages later on

for stage, color in stages.items():
    share = [r for r in run if r["stage"]==stage]
    bars = plot.barh([r["substep"] for r in share], [float(r["ms"]) for r in share], color = color, label = stage)
    plot.bar_label(bars, labels = [f"{float(r['ms']):.2f} ms ({float(r['ms']) / total * 100:.1f}%)" for r in share], padding = 3)


plot.xlim(0, max(float(r["ms"]) for r in run) * 1.35) #room for the labels past the longest bar
plot.xlabel("ms per image")
plot.title(f"{PLATFORM}: substep ASYNC latency per image (CUDA events)\n"
             f"batch {first['batch_size']}, step {first.get('step')}, {first.get('optimization')}, {first.get('precision')}, "
             f"{first['num_images']} images, {first['timestamp']}, total {total:.2f} ms/img")
plot.legend() #show what the colors are associated with from stages
plot.xlabel("ms per image") #add label
plot.gca().invert_yaxis() #make the earliest stages from preprocessing show up first top to bottom
plot.show()

In [ ]:
#HARDWARE HELPERS

#hardware graphs
POWER_RAIL = "VDD_GPU_SOC" #tegrastats rails: VDD_GPU_SOC, VDD_CPU_CV, VIN_SYS_5V0
TEMP_SENSOR = "gpu"        #tegrastats sensors: cpu, gpu, soc0, soc1, soc2, tj, ...
RUN = False #set True to make new data

if RUN:
    run([sys.executable, "main.py", "--setting", "hardware"] + harness_args())
    
#helper func that pulls one rail's watts and one sensor's degrees out of a tegrastats line
#e.g. "... VDD_GPU_SOC 5123mW/5000mW ... gpu@41.2C ..." (first mW number = right now)
def parse(line):
    watts = None
    degrees = None
    power = re.search(POWER_RAIL + r" (\d+)mW/", line) #e.g. finds "VDD_GPU_SOC 5123mW/"
    if power:
        watts = int(power.group(1)) / 1000 #mW to W
    temp = re.search(r"\b" + TEMP_SENSOR + r"@(-?[\d.]+)C", line) #e.g. finds "gpu@41.2C"
    if temp:
        degrees = float(temp.group(1))
    return watts, degrees

#every tegrastats reading: (time stamp, watts, degrees)
readings = []
for line in open(BENCH_DIR / "tegrastats.log"):    
    first_word = line.split(" ")[0] #harness writes time.time() at the start of every line
    try:
        t = float(line.split(" ", 1)[0])
    except ValueError: #line doesn't start with a time stamp
        continue
    watts, degrees = parse(line)
    readings.append((t,) + parse(line))
print(f"{len(readings)} tegrastats readings")

#helper func that is True when a hardware row matches the settings (no precision in hardware rows)
def wanted_hw(row):
    return ((BATCH_SIZE is None or row.get("batch_size") == str(BATCH_SIZE)) and (STEP is None or row.get("step") == str(STEP))
            and (OPTIMIZATION is None or row.get("optimization") == str(OPTIMIZATION)) and row.get("monitor") == "tegrastats")

#captures latest full pipeline window and latest window of every stage that match the settings
full = [r for r in csv.DictReader(open(BENCH_DIR / "hw_metrics.csv")) if wanted_hw(r)]
phases = [r for r in csv.DictReader(open(BENCH_DIR / "hw_stages.csv")) if wanted_hw(r)]
if not full or not phases:
    raise ValueError(f"NO HARDWARE RUN MATCHES THE SETTINGS (batch {BATCH_SIZE}, step {STEP}, opt {OPTIMIZATION})")

full = full[-1] #rows are added to the end of the csv, so the last one is the latest run
windows = {} #stage name -> (start, end) of its latest window
for row in phases:
    windows[row["stage"]] = (float(row["start"]), float(row["end"])) #later rows overwrite earlier ones
hw_stages = {"idle": "#8f8d87"}
hw_stages.update(stages)

#helper func that returns the watts and degrees read inside one window
def inside(start, end):
    watts = []
    degrees = []
    for t, w, d in readings:
        if start <= t <= end:
            if w is not None:
                watts.append(w)
            if d is not None:
                degrees.append(d)
    return watts, degrees

#helper func that turns the settings cell into harness cli args, only the ones that are set (None = harness default)
def harness_args(batch_size = BATCH_SIZE):
    args = []
    if batch_size is not None:
        args = args + ["--batch_size", str(batch_size)]
    if STEP is not None:
        args = args + ["--step", str(STEP)]
    if OPTIMIZATION is not None:
        args = args + ["--optimization", str(OPTIMIZATION)]
    if PRECISION is not None:
        args = args + ["--precision", PRECISION]
    return args

#helper func that runs a command from the harness folder and prints the end of its output
def run(cmd):
    print("  $ " + " ".join(str(c) for c in cmd))
    print("  running... (no output until it finishes)")
    result = subprocess.run([str(c) for c in cmd], cwd = HARNESS_DIR, capture_output = True, text = True)
    print("\n".join(result.stdout.splitlines()[-15:])) #last 15 lines of the harness output
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"COMMAND FAILED (return code {result.returncode})")

In [0]:
#NOTE: CREATE TWO CHARTS THAT SHOW RANGE OF POWER PER SUBSATAGE TESTED AND TEMPERATURE PER STAGE VS IDLE

names = [s for s in hw_stages if s in windows] #idle, pre, inference, post (the ones recorded)
                                               #list of stages to plot
plot.figure(figsize = (12, 4) ) #create a 12 x 4 figure that can fit two graphs 

#power per stage chart goes on left side of figure
plot.subplot(1, 2, 1)
for x, name in enumerate(names):
    watts = inside(*windows[name])[0] #every power reading inside this stage's window
    avg = sum(watts) / len(watts)
    below = max(avg - min(watts), 0) #length of the line below the dot, never negative (steady readings can round avg a hair past min/max)
    above = max(max(watts) - avg, 0) #length of the line above the dot
    plot.errorbar(x, avg, yerr = [[below], [above]], fmt = "o", color = hw_stages[name], capsize = 6, markersize = 8) #dot = avg, line = min to max
    plot.text(x + 0.08, avg, f"{avg:.1f} W", va = "center", fontsize = 9) #average next to the dot
    plot.text(x - 0.08, max(watts), f"max {max(watts):.1f}", ha = "right", va = "bottom", fontsize = 8, color = "#52514e") #max just above the top cap
    plot.text(x - 0.08, min(watts), f"min {min(watts):.1f}", ha = "right", va = "top", fontsize = 8, color = "#52514e") #min just below the bottom cap
plot.xticks(range(len(names)), names)
plot.xlim(-0.7, len(names) - 0.2) #room for the first min/max labels and the last avg label
plot.ylabel(f"{POWER_RAIL} (W)")
plot.title("power per stage", fontsize = 10)

#temp per stage chart goes on right side of figure
plot.subplot(1, 2, 2)
for x, name in enumerate(names):
    degrees = inside(*windows[name])[1] #every temperature reading inside this stage's window
    avg = sum(degrees) / len(degrees)
    below = max(avg - min(degrees), 0)
    above = max(max(degrees) - avg, 0)
    plot.errorbar(x, avg, yerr = [[below], [above]], fmt = "o", color = hw_stages[name], capsize = 6, markersize = 8)
    plot.text(x + 0.08, avg, f"{avg:.1f} C", va = "center", fontsize = 9)
    plot.text(x - 0.08, max(degrees), f"max {max(degrees):.1f}", ha = "right", va = "bottom", fontsize = 8, color = "#52514e")
    plot.text(x - 0.08, min(degrees), f"min {min(degrees):.1f}", ha = "right", va = "top", fontsize = 8, color = "#52514e")
plot.xticks(range(len(names)), names)
plot.xlim(-0.7, len(names) - 0.2)
plot.ylabel(f"{TEMP_SENSOR} (C)")
plot.title("temperature per stage", fontsize = 10)

plot.suptitle(f"{PLATFORM}: hardware range per stage\n"
              f"batch {full['batch_size']}, step {full['step']}, {full['optimization']}", fontsize = 11)
plot.tight_layout()
plot.savefig(GRAPH_DIR / "cuda_hw_range_per_stage.png", dpi = 200, bbox_inches = "tight")
plot.show()

In [ ]:
#NOTE: CREATE TWO CHARTS THAT SHOW POWER AND TEMPERATURE OVER THE WHOLE HARDWARE RUN
#      (full pipeline run through the image db or --step subset, then the idle, pre, inference, post phases)
SMOOTH = False #turn to true if want to smooth graphed data

run_start = float(full["start"]) #time 0 on the x axis = when the full pipeline run started
run_end = 0
for name in windows:
    run_end = max(run_end, windows[name][1]) #end of the last stage phase

#readings from just before the run to just after it, x axis is seconds since the run started
#NOTE: tegrastats isn't recording between the full pipeline run and the stage phases, a nan breaks the line there
#      so it doesn't draw a straight line across the gap
seconds = []
watts = []
degrees = []
last_t = None
for t, w, d in readings:
    if t < run_start - 2 or t > run_end + 2: #outside the run (2 s of margin on each side)
        continue
    if last_t is not None and t - last_t > 1.0: #more than a second with no readings = a gap
        seconds.append(float("nan"))
        watts.append(float("nan"))
        degrees.append(float("nan"))
    if w is None: #rail missing on this line
        w = float("nan")
    if d is None: #sensor missing o this line
        d = float("nan")
    seconds.append(t - run_start)
    watts.append(w)
    degrees.append(d)
    last_t = t
if SMOOTH:
    smooth = []
    for i in range(len(degrees)):
        recent = []
        for v in degrees[max(0, i - 9): i + 1]:
            if not math.isnan(v):
                recent.append(v)
        if math.isnan(degrees[i]) or not recent:
            smooth.append(float("nan")) #keep the gaps
        else:
            smooth.append(sum(recent) / len(recent))

plot.figure(figsize = (12, 6)) #create a 12 x 6 figure that can fit two graphs stacked

#power over time chart goes on top
plot.subplot(2, 1, 1)
plot.axvspan(0, float(full["end"]) - run_start, color = "#c9c7c0", alpha = 0.35, label = "full pipeline") #shade the full pipeline run
for name in windows:
    start, end = windows[name]
    plot.axvspan(start - run_start, end - run_start, color = hw_stages[name], alpha = 0.25, label = name) #shade each stage phase
plot.plot(seconds, watts, color = "#0b0b0b", linewidth = 1.2)
plot.ylabel(f"{POWER_RAIL} (W)")
plot.legend(fontsize = 8, ncol = 5, loc = "lower left", bbox_to_anchor = (0, 1.0), frameon = False) #legend above the chart so it doesn't cover the data

#temperature over time chart goes on the bottom
plot.subplot(2, 1, 2)
plot.axvspan(0, float(full["end"]) - run_start, color = "#c9c7c0", alpha = 0.35)
for name in windows:
    start, end = windows[name]
    plot.axvspan(start - run_start, end - run_start, color = hw_stages[name], alpha = 0.25)
plot.plot(seconds, degrees, color = "#0b0b0b", linewidth = 1.2)
plot.ylabel(f"{TEMP_SENSOR} (C)")
plot.xlabel("seconds since the full pipeline run started")
plot.suptitle(f"{PLATFORM}: hardware run over time\n"
              f"batch {full['batch_size']}, step {full['step']}, {full['optimization']}", fontsize = 11)
plot.tight_layout()
plot.savefig(GRAPH_DIR / "cuda_hw_over_time.png", dpi = 200, bbox_inches = "tight")
plot.show()

In [ ]:
#NOTE: CREATE A CHART THAT SHOWS HOW BATCH SIZE CHANGES END TO END LATENCY AND ENERGY PER IMAGE
#      uses each batch size's latest e2e run (BATCH_MEDIAN row) and latest hardware run, BATCH_SIZE setting is ignored here
#      energy per image = average power while the stages ran x e2e ms per image (W x ms = mJ)
#      average power is weighted by how long one iteration of each stage takes, so slow stages count more
#NOTE: needs a hardware run for every batch size (fill_logs.py does them), batch sizes without one are left off the energy graph

RUN = False #set True to make new e2e data (one e2e run per batch size, about a minute each)
SWEEP = [1, 2, 4, 8, 16, 32]
if RUN:
    for b in SWEEP:
        run([sys.executable, "main.py", "--setting", "e2e"] + harness_args(b))
    rows = list(csv.DictReader(open(BENCH_DIR / "stage_timings.csv"))) #re-read the csv with the new runs in it

#latest BATCH_MEDIAN of every batch size, matching the step / optimization / precision settings
per_image = {} #batch size -> ms per image
for r in rows:
    if r["pass"] != "e2e" or r["substep"] != "BATCH_MEDIAN":
        continue
    if STEP is not None and r["step"] != str(STEP):
        continue
    if OPTIMIZATION is not None and r["optimization"] != str(OPTIMIZATION):
        continue
    if PRECISION is not None and r["precision"] != PRECISION:
        continue
    b = int(r["batch_size"])
    per_image[b] = float(r["ms"]) / b #e2e rows are ms per batch, later rows replace earlier ones
if not per_image:
    raise ValueError("NO e2e RUNS MATCH THE SETTINGS")

#latest hardware stage windows of every batch size (no precision column in hw_stages.csv)
windows_by_batch = {} #batch size -> {stage: (start, end, iterations)}
for r in csv.DictReader(open(BENCH_DIR / "hw_stages.csv")):
    if r["monitor"] != "tegrastats" or r["stage"] == "idle":
        continue
    if STEP is not None and r["step"] != str(STEP):
        continue
    if OPTIMIZATION is not None and r["optimization"] != str(OPTIMIZATION):
        continue
    b = int(r["batch_size"])
    if b not in windows_by_batch:
        windows_by_batch[b] = {}
    windows_by_batch[b][r["stage"]] = (float(r["start"]), float(r["end"]), int(r["iterations"])) #later rows replace earlier ones

#energy per image of every batch size that has both runs
energy = {} #batch size -> mJ per image
for b in per_image:
    if b not in windows_by_batch:
        continue
    watt_time = 0.0 #sum of stage power x one iteration's time
    time_total = 0.0 #sum of one iteration's time
    for start, end, iterations in windows_by_batch[b].values():
        watts = []
        for t, w, d in readings:
            if start <= t <= end:
                watts.append(w)
        if not watts: #no tegrastats readings in this windo
            continue
        iteration_time = (end - start) / iterations
        watt_time += sum(watts) / len(watts) * iteration_time
        time_total += iteration_time
    if time_total > 0:
        energy[b] = watt_time / time_total * per_image[b] #average W x ms per image = mJ per image

missing = []
for b in sorted(per_image):
    if b not in energy:
        missing.append(b)
if missing:
    print(f"NOTE: NO HARDWARE READINGS FOR BATCH {missing}, LEFT OFF THE ENERGY GRAPH (run fill_logs.py, then rerun the hardware helpers cell)")

#settings for the title, leaving out the ones set to None (any)
settings = []
if STEP is not None:
    settings.append(f"step {STEP}")
if OPTIMIZATION is not None:
    settings.append(str(OPTIMIZATION))
if PRECISION is not None:
    settings.append(PRECISION)

plot.figure(figsize = (12, 4))

#left: latency per image
plot.subplot(1, 2, 1)
batches = sorted(per_image)
ms_per_image = []
for b in batches:
    ms_per_image.append(per_image[b])
plot.plot(batches, ms_per_image, marker = "o", color = "#2a78d6")
for b, v in zip(batches, ms_per_image):
    plot.text(b, v + max(ms_per_image) * 0.03, f"{v:.2f}", ha = "center", va = "bottom", fontsize = 9) #value just above each point
plot.xscale("log", base = 2) #batch sizes double, so space them evenl
plot.xticks(batches, batches)
plot.ylim(0, max(ms_per_image) * 1.2) #start at 0, room for the labels
plot.xlabel("batch size")
plot.ylabel("ms per image")
plot.title("end to end latency per image", fontsize = 10)

#right: energy per image
plot.subplot(1, 2, 2)
energy_batches = sorted(energy)
mj_per_image = []
for b in energy_batches:
    mj_per_image.append(energy[b])
if mj_per_image:
    plot.plot(energy_batches, mj_per_image, marker = "o", color = "#52514e")
    for b, v in zip(energy_batches, mj_per_image):
        plot.text(b, v + max(mj_per_image) * 0.03, f"{v:.1f}", ha = "center", va = "bottom", fontsize = 9)
    plot.ylim(0, max(mj_per_image) * 1.2)
plot.xscale("log", base = 2)
plot.xticks(batches, batches) #same batch sizes as the left graph so they line up
plot.xlabel("batch size")
plot.ylabel(f"mJ per image ({POWER_RAIL})")
plot.title("energy per image", fontsize = 10)

plot.suptitle(f"{PLATFORM}: batch size sweep, {', '.join(settings) or 'latest run of every batch size'}", fontsize = 11)
plot.tight_layout(rect = (0, 0, 1, 0.95)) #leave room for the suptitle
plot.savefig(GRAPH_DIR / "cuda_batch_sweep.png", dpi = 200, bbox_inches = "tight")
plot.show()